# Análise Exploratória de Dados (EDA) - NSL-KDD

Análise exploratória da base de dados NSL-KDD para entendimento das características do tráfego de rede e preparação das variáveis.

## 1. Visão Geral do Dataset

Registros de conexões de rede TCP/IP classificados entre normal e ataque.

Bases de dados:
KDDTrain+.txt: Treinamento com 125.973 registros.
KDDTest+.txt: Teste com 22.544 registros.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Carregamento dos Dados

Definição dos 43 nomes de colunas conforme documentação do NSL-KDD.

In [ ]:
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", 
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", 
    "logged_in", "num_compromised", "root_shell", "su_attempted", 
    "num_root", "num_file_creations", "num_shells", "num_access_files", 
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count", 
    "srv_count", "srv_serror_rate", "srv_rerror_rate", "serror_rate", 
    "rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", 
    "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", 
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", 
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", 
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", 
    "dst_host_srv_rerror_rate", "class", "difficulty_level"
]

train_path = os.path.join("..", "..", "data", "raw", "KDDTrain+.txt")
test_path = os.path.join("..", "..", "data", "raw", "KDDTest+.txt")

df_train = pd.read_csv(train_path, header=None, names=columns)
df_test = pd.read_csv(test_path, header=None, names=columns)

print(f"Treino carregado: {df_train.shape[0]} linhas, {df_train.shape[1]} colunas.")
print(f"Teste carregado: {df_test.shape[0]} linhas, {df_test.shape[1]} colunas.")

## 3. Tipos de Variáveis

Identificação das colunas numéricas e categóricas.

In [ ]:
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_train.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Variáveis numéricas ({len(num_cols)}): {num_cols[:5]}")
print(f"Variáveis categóricas ({len(cat_cols)}): {cat_cols}")

## 4. Verificação de Qualidade (Nulos e Duplicados)

Verificação de valores ausentes e registros duplicados.

In [ ]:
nulls_train = df_train.isnull().sum().sum()
nulls_test = df_test.isnull().sum().sum()
dups_train = df_train.duplicated().sum()
dups_test = df_test.duplicated().sum()

print(f"Nulos no Treino: {nulls_train} | Teste: {nulls_test}")
print(f"Duplicados no Treino: {dups_train} | Teste: {dups_test}")

## 5. Análise da Variável Alvo (Target)

Conversão da coluna class em 0 (Normal) e 1 (Ataque) e contagem da distribuição.

In [ ]:
df_train["target"] = df_train["class"].apply(lambda x: 0 if x == "normal" else 1)
df_test["target"] = df_test["class"].apply(lambda x: 0 if x == "normal" else 1)

counts = df_train["target"].value_counts()
pcts = df_train["target"].value_counts(normalize=True) * 100

print(f"Normal (0): {counts[0]} ({pcts[0]:.2f}%)")
print(f"Ataque (1): {counts[1]} ({pcts[1]:.2f}%)")

plt.figure(figsize=(6, 4))
sns.barplot(x=["Normal (0)", "Ataque (1)"], y=[counts[0], counts[1]], palette=["#2ecc71", "#e74c3c"])
plt.title("Distribuição da Classe Alvo no Treino")
plt.ylabel("Quantidade de Conexões")
os.makedirs("imagens", exist_ok=True)
plt.savefig("imagens/distribuicao_target.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Distribuição das Variáveis de Volume

Aplicação da transformação logarítmica log10(x + 1) para ajuste da escala de bytes.

In [ ]:
df_train["log_src_bytes"] = np.log10(df_train["src_bytes"] + 1)
df_train["log_dst_bytes"] = np.log10(df_train["dst_bytes"] + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_train["log_src_bytes"], bins=30, kde=True, ax=axes[0], color="#3498db")
axes[0].set_title("Distribuição de Bytes Enviados (log10)")

sns.histplot(df_train["log_dst_bytes"], bins=30, kde=True, ax=axes[1], color="#9b59b6")
axes[1].set_title("Distribuição de Bytes Recebidos (log10)")

plt.savefig("imagens/distribuicao_variaveis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Análise de Correlação

Matriz de correlação de Pearson para checar atributos correlacionados.

In [ ]:
corr_matrix = df_train[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.iloc[:15, :15], annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Matriz de Correlação")
plt.savefig("imagens/matriz_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Conclusões da Análise Exploratória

1. Base de dados sem valores nulos nem duplicados.
2. Distribuição da classe alvo equilibrada (53,5% Normal vs 46,5% Ataque).
3. Atributos de volume de bytes com transformação logarítmica log10(x + 1).
4. Seleção de atributos no pré-processamento devido à alta correlação entre taxas de erro.